In [0]:
# ── Configuration ──────────────────────────────────────────────────────────────
dbutils.widgets.text("CATALOG", "dev")
CATALOG = dbutils.widgets.get("CATALOG")

In [0]:
# ── Step 1: Read Bronze, Cast Types, Drop Metadata Columns ──
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def typed(name, dtype="string"):
    """Safe cast: blank -> NULL, unparseable -> NULL (never throws). Returns a column named `name`."""
    if dtype == "string":
        return F.col(name).cast("string").alias(name)
    return F.expr(f"try_cast(nullif(trim(cast(`{name}` as string)), '') as {dtype})").alias(name)

bronze_df = spark.table(f"{CATALOG}.bronze.part")

# Metadata columns (_ingestion_timestamp, _source_file, _pipeline_run_id) are NOT selected, so they are dropped.
# _row_seq remembers the source order so "keep the first occurrence" is deterministic on every rerun.
typed_df = (bronze_df.select(
    typed("p_partkey", "int"),
    typed("p_name", "string"),
    typed("p_mfgr", "string"),
    typed("p_brand", "string"),
    typed("p_type", "string"),
    typed("p_size", "int"),
    typed("p_container", "string"),
    typed("p_retailprice", "decimal(18,2)"),
    typed("p_comment", "string")
    )
    .withColumn("_row_seq", F.monotonically_increasing_id()))

bronze_count = bronze_df.count()
print(f"Bronze part rows read: {bronze_count}")
display(typed_df.drop("_row_seq"))

In [0]:
# ── Step 2: Primary Key Validation ──
# Rule: p_partkey must be NOT NULL and UNIQUE.
# Failure codes: NULL_PK, DUPLICATE_PK
# First occurrence (by source order) stays in Silver; later repeats are quarantined.

pk_null = F.col("p_partkey").isNull()
pk_window = Window.partitionBy("p_partkey").orderBy("_row_seq")      # NO "desc" and NO running count: plain row_number

flagged = (typed_df
    .withColumn("is_null_pk", pk_null)
    .withColumn("_pk_rn", F.when(pk_null, F.lit(0)).otherwise(F.row_number().over(pk_window)))
    .withColumn("is_duplicate_pk", F.col("_pk_rn") > 1))

print(f"NULL_PK failures:      {flagged.filter('is_null_pk').count()}")
print(f"DUPLICATE_PK failures: {flagged.filter('is_duplicate_pk').count()}")

In [0]:
# ── Step 3: Mandatory Attribute Validation ──
# Rule: p_name, p_mfgr and p_brand are mandatory descriptive attributes.
# NULL, empty, or whitespace-only values are invalid.
# Failure codes: MISSING_NAME, MISSING_MFGR, MISSING_BRAND

def missing(col):
    """True when the value is NULL, empty, or only whitespace."""
    return F.col(col).isNull() | (F.length(F.trim(F.col(col))) == 0)

flagged = (flagged
    .withColumn("is_missing_name",  missing("p_name"))
    .withColumn("is_missing_mfgr", missing("p_mfgr"))
    .withColumn("is_missing_brand", missing("p_brand")))

print(f"MISSING_NAME failures:  {flagged.filter('is_missing_name').count()}")
print(f"MISSING_MFGR failures:  {flagged.filter('is_missing_mfgr').count()}")
print(f"MISSING_BRAND failures: {flagged.filter('is_missing_brand').count()}")

In [0]:
# ── Step 4: Numeric Range Validation ──
# Rule: p_size (product size) must be strictly positive (> 0).
# Rule: p_retailprice is a monetary value and must be strictly positive (> 0).
# Zero, negative, or unparseable (NULL after casting) values are invalid.
# Failure codes: INVALID_SIZE, INVALID_RETAILPRICE

flagged = (flagged
    .withColumn("is_invalid_size",  F.col("p_size").isNull() | (F.col("p_size") <= 0))
    .withColumn("is_invalid_price", F.col("p_retailprice").isNull() | (F.col("p_retailprice") <= 0)))

print(f"INVALID_SIZE failures:        {flagged.filter('is_invalid_size').count()}")
print(f"INVALID_RETAILPRICE failures: {flagged.filter('is_invalid_price').count()}")

In [0]:
# ── Step 5: Build Rejection Reasons, Split Clean vs Quarantine, Write Tables ──
# NOTE: part sits at the TOP of the hierarchy (parent of lineitem/partsupp) — it has no FK parents to validate.
flagged = flagged.withColumn(
    "rejection_reason",
    F.concat_ws(", ",
        F.when(F.col("is_null_pk"), F.lit("NULL_PK")),
        F.when(F.col("is_duplicate_pk"), F.lit("DUPLICATE_PK")),
        F.when(F.col("is_missing_name"), F.lit("MISSING_NAME")),
        F.when(F.col("is_missing_mfgr"), F.lit("MISSING_MFGR")),
        F.when(F.col("is_missing_brand"), F.lit("MISSING_BRAND")),
        F.when(F.col("is_invalid_size"), F.lit("INVALID_SIZE")),
        F.when(F.col("is_invalid_price"), F.lit("INVALID_RETAILPRICE"))
    ),
)

out_cols = ["p_partkey", "p_name", "p_mfgr", "p_brand", "p_type", "p_size", "p_container", "p_retailprice", "p_comment"]

silver_df     = flagged.filter(F.col("rejection_reason") == "").select(*out_cols)
quarantine_df = flagged.filter(F.col("rejection_reason") != "").select(*out_cols, "rejection_reason")

silver_count = silver_df.count()
quarantine_count = quarantine_df.count()
print(f"Silver rows:     {silver_count}")
print(f"Quarantine rows: {quarantine_count}")

# overwrite => rerunning gives the same result
(silver_df.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.part"))
(quarantine_df.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.part_quarantine"))

print("Tables written: silver.part, silver.part_quarantine")

In [0]:
# ── Step 6: Bronze-Silver-Quarantine Reconciliation ──
bronze_total     = spark.table(f"{CATALOG}.bronze.part").count()
silver_total     = spark.table(f"{CATALOG}.silver.part").count()
quarantine_total = spark.table(f"{CATALOG}.silver.part_quarantine").count()

print(f"Bronze row count:      {bronze_total}")
print(f"Silver row count:      {silver_total}")
print(f"Quarantine row count:  {quarantine_total}")
print(f"Silver + Quarantine:   {silver_total + quarantine_total}")

assert bronze_total == silver_total + quarantine_total, (
    f"RECONCILIATION FAILED: Bronze ({bronze_total}) != Silver ({silver_total}) + Quarantine ({quarantine_total})"
)
print("Reconciliation PASSED: Bronze = Silver + Quarantine")

In [0]:
# ── Step 7: Data Quality Report ──
# Every check is listed, even when zero rows fail.
rules = [
    ("NULL_PK", "p_partkey must not be null", "is_null_pk"),
    ("DUPLICATE_PK", "p_partkey must be unique (first occurrence kept)", "is_duplicate_pk"),
    ("MISSING_NAME", "p_name must not be null/blank", "is_missing_name"),
    ("MISSING_MFGR", "p_mfgr must not be null/blank", "is_missing_mfgr"),
    ("MISSING_BRAND", "p_brand must not be null/blank", "is_missing_brand"),
    ("INVALID_SIZE", "p_size must be strictly positive (> 0)", "is_invalid_size"),
    ("INVALID_RETAILPRICE", "p_retailprice must be strictly positive (> 0)", "is_invalid_price")
]
dq_rows = [("part", code, desc, flagged.filter(F.col(flag)).count(), "Quarantine") for code, desc, flag in rules]

dq_report = spark.createDataFrame(
    dq_rows, ["table", "rule_code", "check_description", "failing_rows", "action_taken"]
)
display(dq_report)

# Persist so the whole run is auditable (overwrite per table is not needed: one table per notebook)
(dq_report.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.dq_report_part"))

if quarantine_count > 0:
    print(f"\n── Quarantined Records ({quarantine_count} rows) ──")
    display(spark.table(f"{CATALOG}.silver.part_quarantine"))
else:
    print("\nNo quarantined records — all rows passed every check.")

print(f"\n── Silver part ({silver_count} rows) ──")
display(spark.table(f"{CATALOG}.silver.part").limit(20))